# 🤟 BISINDO Sign Language Translator
## Real-time Indonesian Sign Language Recognition dengan MediaPipe + LSTM

**Deskripsi Project:**  
Model deep learning yang mengenali bahasa isyarat Indonesia (BISINDO) secara real-time menggunakan landmark tangan dari MediaPipe, diproses oleh LSTM untuk klasifikasi temporal.

**Pendekatan:**
```
Webcam → MediaPipe Hands (21 landmark) → Sequence LSTM → Label BISINDO
```

**Kenapa MediaPipe + LSTM, bukan CNN langsung?**
- MediaPipe mengekstrak 21 titik koordinat tangan (bukan raw pixel) → lebih ringan
- LSTM menangkap pola temporal (gerakan tangan) → lebih akurat untuk gestur dinamis
- Tidak butuh GPU besar saat inference → bisa jalan di laptop biasa
- Real-time ready untuk Streamlit webcam

**Dataset:**  
[Indonesian Sign Language BISINDO - Kaggle](https://www.kaggle.com/datasets/kelsha/indonesian-hand-sign-language-bisindo-dataset)

**Dampak Sosial:**  
Membantu komunikasi antara komunitas tuli/bisu dengan masyarakat umum di Indonesia — menjembatani gap aksesibilitas yang nyata.

---
> ⚡ Project ini menggunakan pipeline yang berbeda dari NutriLens — MediaPipe untuk feature extraction, LSTM untuk sequence modeling, menunjukkan versatilitas dalam computer vision.

## 📦 1. Install & Import Library

In [ ]:
# Install mediapipe (belum ada di Kaggle default)
!pip install mediapipe -q

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import json
import pickle
from pathlib import Path
from tqdm import tqdm
sns.set_style('darkgrid')

import mediapipe as mp

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, BatchNormalization,
    Bidirectional, Input, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
tf.random.set_seed(RNG_SEED)

# MediaPipe setup
mp_hands    = mp.solutions.hands
mp_drawing  = mp.solutions.drawing_utils
mp_styles   = mp.solutions.drawing_styles

print('✅ Imports OK')
print('TensorFlow:', tf.__version__)
print('MediaPipe: ', mp.__version__)

## 🗂️ 2. Load Dataset

**Cara mendapatkan dataset di Kaggle:**
```
Tambahkan dataset:
https://www.kaggle.com/datasets/kelsha/indonesian-hand-sign-language-bisindo-dataset
```

**Struktur folder yang diharapkan:**
```
bisindo/
├── A/
│   ├── img001.jpg
│   └── ...
├── B/
└── ...
```

In [ ]:
# ============================================================
# PATH — sesuaikan dengan struktur dataset Kaggle
# ============================================================
DATASET_DIR = '/kaggle/input/indonesian-hand-sign-language-bisindo-dataset/'

# Cek struktur folder
print('Struktur dataset:')
classes = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])
print(f'Kelas ({len(classes)}):', classes)

# Build dataframe
filepaths, labels = [], []
for cls in classes:
    cls_dir = os.path.join(DATASET_DIR, cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            filepaths.append(os.path.join(cls_dir, f))
            labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
print(f'\nTotal gambar: {len(df)}')
print('\nDistribusi kelas:')
print(df['label'].value_counts())

In [ ]:
# Visualisasi distribusi kelas
plt.figure(figsize=(14, 4))
sns.countplot(data=df, x='label', palette='viridis', order=sorted(df['label'].unique()))
plt.title('Distribusi Kelas BISINDO', fontsize=13)
plt.xlabel('Kelas')
plt.ylabel('Jumlah Gambar')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## ✋ 3. MediaPipe — Ekstraksi Landmark Tangan

MediaPipe Hands mendeteksi **21 titik landmark** pada tangan:
- Setiap titik punya koordinat (x, y, z)
- Total fitur per frame: 21 × 3 = **63 nilai**
- Jauh lebih efisien daripada raw pixel (300×300×3 = 270,000)

```
Titik landmark:
  0: Wrist
  1-4: Ibu jari
  5-8: Telunjuk
  9-12: Jari tengah
  13-16: Jari manis
  17-20: Kelingking
```

In [ ]:
NUM_LANDMARKS = 21
NUM_COORDS    = 3   # x, y, z
FEATURE_SIZE  = NUM_LANDMARKS * NUM_COORDS  # 63

def extract_landmarks(image_path, hands_detector):
    """
    Ekstrak 21 landmark tangan dari gambar.
    Return: array (63,) atau None jika tangan tidak terdeteksi.
    """
    img = cv2.imread(image_path)
    if img is None:
        return None

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands_detector.process(img_rgb)

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]  # ambil tangan pertama
        landmarks = []
        for lm in hand_landmarks.landmark:
            landmarks.extend([lm.x, lm.y, lm.z])
        return np.array(landmarks, dtype=np.float32)
    return None


def normalize_landmarks(landmarks):
    """
    Normalisasi landmark relatif terhadap wrist (titik 0)
    agar posisi tangan di frame tidak mempengaruhi prediksi.
    """
    reshaped = landmarks.reshape(NUM_LANDMARKS, NUM_COORDS)
    wrist    = reshaped[0]  # titik 0 = wrist
    normalized = reshaped - wrist  # relatif ke wrist

    # Scale agar invariant terhadap ukuran tangan
    scale = np.max(np.abs(normalized)) + 1e-8
    normalized = normalized / scale
    return normalized.flatten()


print(f'Feature size per frame: {FEATURE_SIZE}')
print('✅ Fungsi ekstraksi landmark siap')

In [ ]:
# Preview — visualisasi landmark pada beberapa gambar
sample_classes = random.sample(classes, min(6, len(classes)))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
) as hands:
    for i, cls in enumerate(sample_classes):
        cls_dir   = os.path.join(DATASET_DIR, cls)
        img_files = os.listdir(cls_dir)
        img_path  = os.path.join(cls_dir, random.choice(img_files))

        img     = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(img_rgb)

        if results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                img_rgb,
                results.multi_hand_landmarks[0],
                mp_hands.HAND_CONNECTIONS,
                mp_styles.get_default_hand_landmarks_style(),
                mp_styles.get_default_hand_connections_style()
            )

        axes[i].imshow(img_rgb)
        axes[i].set_title(f'BISINDO: {cls}', fontsize=12, fontweight='bold')
        axes[i].axis('off')

plt.suptitle('Visualisasi MediaPipe Landmark pada BISINDO', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔄 4. Ekstraksi Fitur dari Seluruh Dataset

Karena dataset berupa gambar statis (bukan video), kita simulasikan sequence dengan **augmentasi temporal**:
- Setiap gambar diproses jadi 1 frame
- Dibuat sequence panjang N dengan slight variation (noise kecil)
- LSTM tetap bisa belajar pola spatial dari sequence ini

In [ ]:
SEQUENCE_LENGTH = 10  # panjang sequence per sample
NOISE_STD       = 0.01  # noise untuk simulasi temporal variation

X_raw, y_raw = [], []
skipped      = 0

with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
) as hands:
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Ekstraksi landmark'):
        landmarks = extract_landmarks(row['filepath'], hands)

        if landmarks is None:
            skipped += 1
            continue

        # Normalisasi
        landmarks_norm = normalize_landmarks(landmarks)

        # Buat sequence: 1 landmark + slight noise untuk SEQUENCE_LENGTH frames
        sequence = []
        for _ in range(SEQUENCE_LENGTH):
            noise  = np.random.normal(0, NOISE_STD, landmarks_norm.shape).astype(np.float32)
            frame  = np.clip(landmarks_norm + noise, -1, 1)
            sequence.append(frame)

        X_raw.append(sequence)
        y_raw.append(row['label'])

X_raw = np.array(X_raw, dtype=np.float32)  # (N, SEQUENCE_LENGTH, 63)
y_raw = np.array(y_raw)

print(f'\n✅ Ekstraksi selesai')
print(f'Total sample   : {len(X_raw)}')
print(f'Gambar dilewati: {skipped} (tangan tidak terdeteksi)')
print(f'Shape X        : {X_raw.shape}  → (samples, sequence, features)')
print(f'Shape y        : {y_raw.shape}')

## 🏷️ 5. Label Encoding & Split Dataset

In [ ]:
# Label encoding
le          = LabelEncoder()
y_encoded   = le.fit_transform(y_raw)
CLASS_NAMES = le.classes_.tolist()
NUM_CLASSES = len(CLASS_NAMES)

print(f'Jumlah kelas: {NUM_CLASSES}')
print(f'Kelas: {CLASS_NAMES}')

# One-hot
y_onehot = to_categorical(y_encoded, num_classes=NUM_CLASSES)

# Split 80/10/10
X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw, y_onehot, test_size=0.2, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42,
    stratify=np.argmax(y_temp, axis=1)
)

print(f'\nTrain : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')

## 🏗️ 6. Bangun Model Bidirectional LSTM

**Kenapa Bidirectional LSTM?**
- LSTM biasa hanya baca sequence dari kiri ke kanan
- Bidirectional baca dari dua arah → menangkap konteks lebih kaya
- Untuk gestur tangan, informasi awal DAN akhir gerakan sama pentingnya

In [ ]:
def build_bilstm_model(sequence_length, feature_size, num_classes):
    inputs = Input(shape=(sequence_length, feature_size))

    # Layer 1 — Bidirectional LSTM
    x = Bidirectional(LSTM(128, return_sequences=True))(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Layer 2 — Bidirectional LSTM
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Layer 3 — LSTM
    x = LSTM(32, return_sequences=False)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    # Dense head
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='BISINDO_BiLSTM')
    return model

model = build_bilstm_model(SEQUENCE_LENGTH, FEATURE_SIZE, NUM_CLASSES)

model.compile(
    optimizer=Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

model.summary()
print(f'\nTotal parameter: {model.count_params():,}')

## ⚙️ 7. Training

In [ ]:
callbacks = [
    ModelCheckpoint(
        'best_bisindo_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

print('Mulai Training...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Plot training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Accuracy')
axes[1].legend()

plt.suptitle('Training BISINDO BiLSTM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 📈 8. Evaluasi Model

In [ ]:
# Load best model
best_model = tf.keras.models.load_model('best_bisindo_model.keras')

# Prediksi test set
y_true = np.argmax(y_test, axis=1)
y_prob = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig_size = max(10, len(CLASS_NAMES) // 2)
plt.figure(figsize=(fig_size, fig_size))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)
disp.plot(cmap='Blues', values_format='d', ax=plt.gca())
plt.title('Confusion Matrix — BISINDO Sign Language', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Akurasi per kelas
per_class_acc = cm.diagonal() / cm.sum(axis=1)
per_class_df  = pd.DataFrame({
    'Kelas':   CLASS_NAMES,
    'Akurasi': per_class_acc
}).sort_values('Akurasi', ascending=True)

plt.figure(figsize=(10, max(5, len(CLASS_NAMES) // 3)))
colors = ['#e74c3c' if a < 0.75 else '#f39c12' if a < 0.85 else '#27ae60'
          for a in per_class_df['Akurasi']]
plt.barh(per_class_df['Kelas'], per_class_df['Akurasi'], color=colors)
plt.axvline(0.8, color='red', linestyle='--', label='Target 80%')
plt.xlabel('Akurasi')
plt.title('Akurasi per Kelas BISINDO', fontsize=13)
plt.xlim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

print(per_class_df.sort_values('Akurasi', ascending=False).to_string(index=False))

## 🧪 9. Inference Test — Simulasi Real-time

In [ ]:
CONFIDENCE_THRESHOLD = 0.60

def predict_sign(landmarks_sequence, model, class_names, threshold=CONFIDENCE_THRESHOLD):
    """
    Prediksi gestur dari sequence landmark.
    Return: dict hasil prediksi.
    """
    seq_array = np.expand_dims(landmarks_sequence, axis=0)  # (1, SEQ, 63)
    probs     = model.predict(seq_array, verbose=0)[0]
    top_idx   = np.argmax(probs)
    top_conf  = float(probs[top_idx])
    top3_idx  = np.argsort(probs)[::-1][:3]

    return {
        'label':        class_names[top_idx] if top_conf >= threshold else '?',
        'confidence':   top_conf,
        'is_confident': top_conf >= threshold,
        'top3': [
            {'label': class_names[i], 'confidence': float(probs[i])}
            for i in top3_idx
        ]
    }


# Test dengan sampel dari test set
sample_idx = random.randint(0, len(X_test) - 1)
result     = predict_sign(X_test[sample_idx], best_model, CLASS_NAMES)
true_label = CLASS_NAMES[y_true[sample_idx]]

print('=== INFERENCE TEST ===')
print(f'Label sebenarnya : {true_label}')
print(f'Prediksi         : {result["label"]}')
print(f'Confidence       : {result["confidence"]:.1%}')
print(f'Terdeteksi yakin : {result["is_confident"]}')
print(f'\nTop-3 prediksi:')
for r in result['top3']:
    print(f'  {r["label"]:5s} → {r["confidence"]:.1%}')

## 💾 10. Simpan Model & Assets untuk Streamlit

In [ ]:
import shutil

# Gunakan best model
shutil.copy('best_bisindo_model.keras', 'bisindo_model.keras')
print('✅ Model disimpan: bisindo_model.keras')

# Simpan LabelEncoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print('✅ Label encoder disimpan: label_encoder.pkl')

# Simpan assets
assets = {
    'class_names':     CLASS_NAMES,
    'num_classes':     NUM_CLASSES,
    'sequence_length': SEQUENCE_LENGTH,
    'feature_size':    FEATURE_SIZE,
    'threshold':       CONFIDENCE_THRESHOLD
}
with open('bisindo_assets.json', 'w') as f:
    json.dump(assets, f, indent=2)
print('✅ Assets disimpan: bisindo_assets.json')

print('\nFile yang dibutuhkan Streamlit:')
print('  → bisindo_model.keras')
print('  → bisindo_assets.json')
print('  → label_encoder.pkl')
print('  → app.py (Streamlit webcam)')

## 📋 11. Ringkasan & Narasi Portofolio

In [ ]:
test_loss, test_acc, test_prec, test_rec, test_auc = best_model.evaluate(
    X_test, y_test, verbose=0
)

print('='*55)
print('   RINGKASAN — BISINDO SIGN LANGUAGE TRANSLATOR')
print('='*55)
print(f'  Arsitektur   : Bidirectional LSTM (3 layer)')
print(f'  Feature Ext  : MediaPipe Hands (21 landmark × 3)')
print(f'  Jumlah Kelas : {NUM_CLASSES}')
print(f'  Test Accuracy: {test_acc:.3f}')
print(f'  Precision    : {test_prec:.3f}')
print(f'  Recall       : {test_rec:.3f}')
print(f'  AUC          : {test_auc:.3f}')
print('='*55)
print()
print('NARASI PORTOFOLIO (Apple Developer Academy):')
print("""
Masalah: Lebih dari 2.5 juta penyandang disabilitas pendengaran
di Indonesia menggunakan BISINDO sebagai bahasa utama mereka,
namun sangat sedikit orang di luar komunitas tuli yang memahaminya.
Gap komunikasi ini menyebabkan isolasi sosial dan keterbatasan
akses terhadap layanan publik.

Solusi: Aplikasi real-time yang menerjemahkan gestur tangan
BISINDO menggunakan webcam biasa — tidak perlu hardware khusus.
Siapapun bisa mulai memahami bahasa isyarat dalam hitungan detik.

Pendekatan Teknis: MediaPipe dipilih untuk ekstraksi landmark
karena lebih robust terhadap variasi pencahayaan dan background
dibanding CNN berbasis pixel. Bidirectional LSTM menangkap pola
temporal gestur dari dua arah, meningkatkan akurasi pada gestur
yang memiliki fase awal dan akhir yang bermakna.

Limitasi & Etika: Model dilatih pada dataset terbatas dan belum
mencakup seluruh kosakata BISINDO. Aplikasi ini adalah alat bantu
komunikasi, bukan pengganti interpreter profesional.

Dampak: Membuka jembatan komunikasi antara komunitas tuli/bisu
dengan masyarakat umum — sejalan dengan nilai aksesibilitas
yang menjadi fondasi teknologi Apple.
""")